<a href="https://colab.research.google.com/github/OhadParis/spatial-llm-orchestrator/blob/main/LiDAR_Orchestrator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q pandas anthropic pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.1/763.1 kB 26.5 MB/s eta 0:00:00


In [4]:
# @title

import pandas as pd
from anthropic import Anthropic
from pydantic import BaseModel, Field
from typing import Optional

CONFIG = {
    "model": "claude-sonnet-4-6", # Easy to change in one place
    "max_tokens": 1024,
    "system_prompt": "You are a LiDAR data assistant. Extract spatial filtering parameters."
}

# 1. Setup your API Key
# Replace 'your-key-here' with your actual Anthropic API Key
client = Anthropic(api_key='KEY')

# 2. Define the "Brain" (The Schema)
# defining what data the LLM must extract.
class SpatialFilter(BaseModel):
    min_x: Optional[float] = Field(None, description="The minimum X coordinate")
    max_x: Optional[float] = Field(None, description="The maximum X coordinate")
    min_z: Optional[float] = Field(None, description="The minimum Z (elevation) coordinate")
    label: str = Field(..., description="A short name for this specific search")

# 3. The Orchestration Function
def get_spatial_params(user_query: str):
    response = client.messages.create(
        model=CONFIG["model"],
        max_tokens=CONFIG["max_tokens"],
        system=CONFIG["system_prompt"],
        messages=[{"role": "user", "content": user_query}]
    )
    return response.content[0].text

# 4. Test it out
user_prompt = "Find all points in the north quadrant where the elevation is higher than 50 meters."
print(f"User Request: {user_prompt}")
print("-" * 30)
print(get_spatial_params(user_prompt))

User Request: Find all points in the north quadrant where the elevation is higher than 50 meters.
------------------------------
## Spatial Filter Parameter Extraction

Here are the extracted parameters from your query:

---

### 📍 Spatial Filter Parameters

| Parameter | Value | Notes |
|-----------|-------|-------|
| **Region/Quadrant** | North | Cardinal direction filter |
| **Elevation Threshold** | > 50 meters | Minimum elevation filter |
| **Filter Type** | Bounding + Attribute | Combined spatial + value filter |

---

### 🔧 Filter Logic

```python
# Extracted Filter Parameters
spatial_filter = {
    "quadrant": "north",
    "direction": "N",
    "boundary": "y > origin_y"      # Points above center Y axis
}

attribute_filter = {
    "field": "elevation",           # Z-value or dedicated elevation field
    "operator": "greater_than",
    "threshold": 50,
    "unit": "meters"
}

combined_logic = "AND"              # Both conditions must be satisfied
```

---

### 🗂️ Query Transla

In [5]:
# @title
import pandas as pd
import json
from anthropic import Anthropic

# 1. Initialize Client
client = Anthropic(api_key='KEY')

# 2. Define the tool (The exact structure we want Claude to return)
spatial_filter_tool = {
    "name": "filter_spatial_data",
    "description": "Extract numerical bounds for filtering spatial LiDAR data from a user query.",
    "input_schema": {
        "type": "object",
        "properties": {
            "min_x": {"type": "number", "description": "Minimum X coordinate limit, if mentioned."},
            "max_x": {"type": "number", "description": "Maximum X coordinate limit, if mentioned."},
            "min_z": {"type": "number", "description": "Minimum Z (elevation) limit, if mentioned."},
            "max_z": {"type": "number", "description": "Maximum Z (elevation) limit, if mentioned."}
        },
        "required": [] # None are strictly required; depends on what the user asks for
    }
}

# 3. The Orchestrator Function
def query_lidar_data(csv_path: str, user_query: str):
    # Load the big data engine
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df):,} LiDAR points successfully.")

    # Ask Claude to parse the query using our tool definitions
    response = client.messages.create(
        model="claude-sonnet-4-6", # Using the stable 3.5 Sonnet ID
        max_tokens=1024,
        system="You are a LiDAR data assistant. Your single task is to extract spatial filters into the tool provided.",
        tools=[spatial_filter_tool],
        tool_choice={"type": "tool", "name": "filter_spatial_data"}, # Force it to use the tool
        messages=[{"role": "user", "content": user_query}]
    )

    # Extract the structured JSON data from Claude's response
    tool_use = response.content[0]
    filters = tool_use.input
    print(f"\nClaude's Extracted Parameters: {json.dumps(filters, indent=2)}")

    # Apply the filters dynamically to the pandas DataFrame
    filtered_df = df.copy()

    if "min_x" in filters and filters["min_x"] is not None:
        filtered_df = filtered_df[filtered_df['X'] >= filters["min_x"]]
    if "max_x" in filters and filters["max_x"] is not None:
        filtered_df = filtered_df[filtered_df['X'] <= filters["max_x"]]
    if "min_z" in filters and filters["min_z"] is not None:
        filtered_df = filtered_df[filtered_df['Z'] >= filters["min_z"]]
    if "max_z" in filters and filters["max_z"] is not None:
        filtered_df = filtered_df[filtered_df['Z'] <= filters["max_z"]]

    print(f"Filtering complete. Remaining points: {len(filtered_df):,}")
    return filtered_df

# 4. Run the Pipeline
     ## Note: the csv below is a small zylindrical pointcloud from within the larger habitat of interest.
     ## The full file is too large to be uploaded to Google Colab.
# Replace with your actual file path and column variations if needed
csv_filename = "PP_CD_DF.csv"
query = "Show me all data points where the elevation is above 80 meters."

filtered_results = query_lidar_data(csv_filename, query)

# === ADD THESE LINES TO PRINT THE ACTUAL DATA ===
print("\n" + "="*40)
print("          FILTERED DATAFRAME ROW PREVIEW          ")
print("="*40)

if not filtered_results.empty:
    # This prints the first 20 rows that matched your criteria
    print(filtered_results.head(20).to_string())

    # Optional: Save the filtered points to a new CSV file so you can download it
    filtered_results.to_csv("filtered_spatial_output.csv", index=False)
    print("\n[Success] Saved filtered rows to 'filtered_spatial_output.csv'")
else:
    print("The dataframe is empty. No rows matched the criteria.")

FileNotFoundError: [Errno 2] No such file or directory: 'PP_CD_DF.csv'